<div style="text-align: center;">

  <img src="img/LogoUesc.png" alt="UESC" style="width: 60px; display: block; margin: 0 auto 5px auto;">

  <div style="font-size: 20px;"><strong>UNIVERSIDADE ESTADUAL DE SANTA CRUZ - UESC</strong></div>
  <div style="font-size: 18px;">DEPARTAMENTO DE ENGENHARIAS E COMPUTAÇÃO - DEC</div>
  <div style="font-size: 18px;">Grupo de Pesquisas em Simulações e Controle de Processos</div>

  <hr style="margin: 20px 0; border: 1px solid #0074B7;">

  <div style="font-size: 22px; font-weight: bold; margin-bottom: 10px;">PROGRAMAÇÃO EM PYTHON PARA AS ENGENHARIAS</div>

  <div style="font-size: 18px; margin-top: 10px;"><strong>Tema</strong>: Previsão da Composição em uma Corrente de Destilação com Redes Neurais</div>
  <div style="font-size: 18px;"><strong>Prof. Dr. E.R.Edwards</strong></div>

  <div style="font-size: 16px; margin-top: 15px;">Ilhéus - BA, Abril de 2025</div>
  
  <hr style="margin: 20px 0; border: 1px solid #0074B7;">

</div>

**__Sumário__**

- [1. Problema de engenharia e dados experimentais](#secao-1)
- [2. Objetivos e arquitetura da rede neural](#secao-2)
- [3. Preparação e normalização dos dados](#secao-3)
- [4. Função de ativação e inicialização dos parâmetros](#secao-4)
- [5. Treinamento por retropropagação](#secao-5)
- [6. Previsões e interpretação dos resultados](#secao-6)

<div id="secao-1" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">
  1. Problema de engenharia e dados experimentais
</div>

__Enunciado__:

Uma planta de destilação de etanol coleta dados operacionais de várias correntes do processo para controlar a qualidade do produto. Os engenheiros estão interessados em prever a fração mássica de etanol em diferentes correntes, com base em variáveis de processo como temperatura e vazão mássica.

A tabela abaixo mostra dados de 4 correntes retiradas da coluna de destilação:

<img src="img/Tab_06.png" alt="UESC" style="width: 550px; display: block; margin: 0 auto 5px auto;">
<p style="text-align: center;"><strong>Figura: Tabela de Coleta de Dados em uma Planta de Destilação de etanol.</strong></p>

<div id="secao-2" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">
  2. Objetivos e arquitetura da rede neural
</div>

**Objetivo:** desenvolver, sem bibliotecas de redes neurais prontas, um modelo para estimar o teor de etanol a partir da temperatura e da vazão mássica.

A arquitetura utilizada terá duas entradas, uma camada oculta com três neurônios ReLU e uma saída linear (regressão). Os valores da tabela estão expressos em **porcentagem (%)**, não em fração adimensional.

Os quatro registros são usados para treinamento; os dois novos pontos serão apenas exemplos de aplicação, não uma validação independente.

**Novas condições:**


<img src="img/Tab_07.png" alt="UESC" style="width: 300px; display: block; margin: 0 auto 5px auto;">
<p style="text-align: center;"><strong>Figura: Tabela de dados para previsão de novas correntes.</strong></p>

4. Interprete criticamente os resultados: quatro amostras são suficientes para demonstrar capacidade de generalização?

<div id="secao-3" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">
  3. Preparação e normalização dos dados
</div>

A normalização mín–máx transforma cada variável para a escala de 0 a 1, considerando os valores mínimo e máximo das **amostras de treinamento**. Para cada variável,

$$x_{norm}=\frac{x-x_{min}}{x_{max}-x_{min}}.$$

Na previsão, utilizamos os mesmos limites calculados no treinamento. Valores novos podem ficar fora de $[0,1]$ quando estão fora da faixa observada. A segunda corrente, com vazão de 1180 kg/h, permanece dentro da faixa de treinamento (1100 a 1350 kg/h).

In [ ]:
import numpy as np

# Função de normalização mín-máx
def normalizar(dados):
    minimo = dados.min(axis=0)
    maximo = dados.max(axis=0)
    return (dados - minimo) / (maximo - minimo), minimo, maximo

# Dados de entrada: Temperatura (°C), Vazão (kg/h)
entradas = np.array([
    [78, 1200],  # Corrente A
    [72, 1350],  # Corrente B
    [80, 1100],  # Corrente C
    [70, 1250],  # Corrente D
], dtype=np.float32)

# Dados de saída: Teor de etanol (%)
saidas = np.array([
    [95],
    [88],
    [97],
    [85],
], dtype=np.float32)

# Normalizando entradas e saídas
entradas_norm, entradas_min, entradas_max = normalizar(entradas)
saidas_norm, saidas_min, saidas_max = normalizar(saidas)


<div id="secao-4" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">
  4. Função de ativação e inicialização dos parâmetros
</div>

A função ReLU é $f(z)=\max(0,z)$. Sua derivada é 1 para $z>0$ e 0 para $z<0$; em $z=0$, adotamos convencionalmente o valor 0. A camada de saída é linear para permitir regressão.


In [ ]:
# Função ReLU e derivada
def relu(x):
    return np.maximum(0, x)

def relu_derivada(x):
    return (x> 0).astype(float)    

**Inicialização dos pesos e vieses.** Uma semente aleatória fixa permite reproduzir os resultados.

In [ ]:
# Inicialização dos pesos.
np.random.seed(42)

# 2 entradas, 3 neurônios na camada oculta, 1 saída.
pesos_entrada_oculta = np.random.randn(2,3)
bias_oculta = np.zeros((1,3))

pesos_oculta_saida = np.random.randn(3, 1)
bias_saida = np.zeros((1,1))

#Hiperparâmetros
taxa_aprendizado = 0.1
epochs = 1000

<div id="secao-5" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">
  5. Treinamento por retropropagação
</div>

O treinamento minimiza o erro quadrático médio (MSE):

$$\mathrm{MSE}=\frac{1}{N}\sum_{i=1}^{N}(y_i-\hat y_i)^2.$$

Na retropropagação, a derivada em relação à previsão é $2(\hat y-y)/N$, com $N$ igual ao número de amostras. Esse fator garante que o gradiente corresponda à **média** dos erros, independentemente da quantidade de dados.


In [ ]:
for epoca in range(epochs):
    # FORWARD
    camada_oculta_input = np.dot(entradas_norm, pesos_entrada_oculta) + bias_oculta
    camada_oculta_saida = relu(camada_oculta_input)

    saida_input = np.dot(camada_oculta_saida, pesos_oculta_saida) + bias_saida
    saida_predita = saida_input  # Sem função de ativação final (regressão)

    # ERRO
    erro = saidas_norm - saida_predita

    # BACKPROPAGATION
    d_saida = -2 * erro / len(entradas_norm)  # derivada do MSE médio

    d_pesos_oculta_saida = np.dot(camada_oculta_saida.T, d_saida)
    d_bias_saida = d_saida.sum(axis=0, keepdims=True)

    d_oculta = np.dot(d_saida, pesos_oculta_saida.T) * relu_derivada(camada_oculta_input)
    d_pesos_entrada_oculta = np.dot(entradas_norm.T, d_oculta)
    d_bias_oculta = d_oculta.sum(axis=0, keepdims=True)

    # ATUALIZAÇÃO
    pesos_oculta_saida -= taxa_aprendizado * d_pesos_oculta_saida
    bias_saida -= taxa_aprendizado * d_bias_saida

    pesos_entrada_oculta -= taxa_aprendizado * d_pesos_entrada_oculta
    bias_oculta -= taxa_aprendizado * d_bias_oculta

    # Exibir erro médio a cada 200 épocas
    if epoca % 200 == 0:
        mse = np.mean(erro**2)
        print(f'Época {epoca}, Erro quadrático médio: {mse:.4f}')

<div id="secao-6" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">
  6. Previsões e interpretação dos resultados
</div>

Os novos pontos são estimados com os parâmetros aprendidos. Como a saída foi normalizada durante o treinamento, é necessário retornar à escala original (%).


In [ ]:
# Novas entradas
novas_entradas = np.array([
    [75, 1300],
    [73, 1180]
], dtype=np.float32)

# Normalizar novas entradas com base nos dados originais
novas_entradas_norm = (novas_entradas - entradas_min) / (entradas_max - entradas_min)

# Forward para previsão
camada_oculta_nova = relu(np.dot(novas_entradas_norm, pesos_entrada_oculta) + bias_oculta)
saida_nova_norm = np.dot(camada_oculta_nova, pesos_oculta_saida) + bias_saida

# Desnormalizar saída
saida_nova = saida_nova_norm * (saidas_max - saidas_min) + saidas_min
print("\nPrevisões para novas correntes:")
for i, pred in enumerate(saida_nova):
    print(f"Corrente {i+1}: Fração de etanol estimada = {pred[0]:.2f}%")


**Interpretação dos resultados.**

As previsões impressas pela célula anterior devem ser utilizadas na discussão, em vez de registrar antecipadamente números que podem mudar após alterações no treinamento. É importante comparar as estimativas com a faixa dos dados originais (85% a 97%) e verificar se o erro de treinamento diminuiu.

**Limitação fundamental:** os quatro registros foram usados para ajustar os parâmetros. Sem um conjunto independente de validação, não é possível afirmar que a rede generaliza para novas condições. Além disso, temperatura e vazão, isoladamente, podem não representar todas as variáveis que determinam a composição em uma coluna de destilação.


**Conclusão didática:** este exercício demonstra a construção explícita de uma rede neural, incluindo normalização, propagação direta, cálculo de gradientes e atualização de pesos. Os resultados são ilustrativos e não constituem um modelo validado para controle de qualidade industrial. Para uso real, seriam necessários mais dados representativos, validação independente e análise das variáveis relevantes do processo.


<div style="text-align: center; font-size: 12px; color: gray;
            margin-top: 40px; padding-top: 15px;
            border-top: 1px solid #dddddd;">

  Material desenvolvido pelo Grupo de Pesquisas em
  Modelagem Computacional da UESC.<br>

  © 2026 — Todos os direitos reservados.
  Uso comercial somente mediante autorização.

</div>